# Sesión 2 · Fuentes y tipos de datos
**AutoGest** · Un dato, tres puertas de entrada + fuentes externas.

Se cargan servicios del taller en CSV, JSON y SQLite; se descargan 1,827 días de clima real de Ibagué desde la API pública open-meteo (estación PERALES) y se unen las fuentes.

In [ ]:
import json, sqlite3, urllib.request
import pandas as pd
from pathlib import Path

# Muestra idéntica en 3 formatos
ventas = pd.DataFrame([...])  # ver carga_fuentes_autogest.py

In [ ]:
RUTA_CSV = Path("datasets/servicios_taller.csv")
RUTA_JSON = Path("datasets/servicios_taller.json")
RUTA_DB = Path("datasets/servicios_taller.db")

for nombre, ruta, lector in [
    ("CSV", RUTA_CSV, lambda r: pd.read_csv(r)),
    ("JSON", RUTA_JSON, lambda r: pd.read_json(r)),
    ("SQLite", RUTA_DB, lambda r: pd.read_sql("SELECT * FROM servicios", sqlite3.connect(r))),
]:
    d = lector(ruta)
    print(nombre, d.shape, len(d.dtypes))

<details><summary><b>Salida ejecutada</b></summary>

```text
[CSV] shape=(5, 5) · [JSON] shape=(5, 5) · [SQLite] shape=(5, 5)
fecha queda inferida como str/texto en los tres formatos porque viene en ISO 'YYYY-MM-DD'.
```
</details>

## Fuente externa · Clima Ibagué (API pública open-meteo)

Cumple el requisito **'al menos una API o BD pública'** del proyecto. Parámetros: lat 4.4219, lon -75.1331 (PERALES), rango 2020-01-01 a 2024-12-31.

In [ ]:
# Fetch API
url = ("https://archive-api.open-meteo.com/v1/archive"
       "?latitude=4.4219&longitude=-75.1331"
       "&start_date=2020-01-01&end_date=2024-12-31"
       "&daily=precipitation_sum,temperature_2m_max,temperature_2m_min"
       "&timezone=America%2FBogota")
with urllib.request.urlopen(url, timeout=60) as r:
    data = json.load(r)

clima = pd.DataFrame(data["daily"])
clima.columns = ["fecha","precipitacion_mm","temp_max_c","temp_min_c"]
clima["es_lluvia_extrema"] = clima["precipitacion_mm"].fillna(0) > 7.5
clima.to_csv("../../data_pipeline/data/raw/clima_ibague_openmeteo.csv", index=False)
print(clima.shape, "días | días lluvia extrema:", int(clima["es_lluvia_extrema"].sum()))

<details><summary><b>Salida ejecutada</b></summary>

```text
Clima descargado: 1,827 días (2020-01-01 a 2024-12-31)
Días con lluvia extrema (>7.5mm): 174
```
</details>

## Unión de fuentes (left join)

Implementa el cruce servicios × clima del pipeline (`data_pipeline/scripts`): **190,414 servicios ocurrieron en días de lluvia extrema** — la señal climática de FASE_1.

In [ ]:
# Left join servicios × clima
servicios = pd.read_csv("../../data_pipeline/data/raw/enhanced_motor_vehicle_repair_towing_dataset.csv",
                   usecols=["Service Type","Repair Date","Total Cost"], low_memory=False)
servicios["fecha"] = pd.to_datetime(servicios["Repair Date"]).dt.date
clima["fecha"] = pd.to_datetime(clima["fecha"]).dt.date
union = servicios.merge(clima, on="fecha", how="left")
print("Filas tras join:", f"{union.shape[0]:,}", "| sin dato climático:", union["precipitacion_mm"].isna().sum())

<details><summary><b>Salida ejecutada</b></summary>

```text
Filas tras join: 2,000,000 | sin dato climático: 0
Servicios en día de lluvia extrema: 190,414
```
</details>